# Pipeline1Completo


## 1. Imports


In [2]:
import sys
!{sys.executable} -m pip install feature_engine --quiet

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

src_path = "C:/Users/maria/OneDrive/Escritorio/INGENIERIA MATEMATICA/4 CARRERA/CUATRI - 2/MODELIZACION EN INGENIERIA DE DATOS/practica1_ingenieria_de_datos/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from preprocessing.practica1_preprocessing import Practica1Preprocessing
from filtering.practica1_filtering import Practica1Filtering

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    average_precision_score,
    classification_report,
)

## 2. Carga de datos


In [3]:
#carga de datos
df_train = pd.read_csv("data/df_train_small.csv")
df_test = pd.read_csv("data/df_test_small.csv")

df_train_raw = df_train.reset_index(drop=True)
df_test_raw = df_test.reset_index(drop=True)

print(f"Train: {df_train_raw.shape}")
print(f"Test:  {df_test_raw.shape}")


Train: (80000, 151)
Test:  (20000, 151)


In [4]:
TARGET = "loan_status"
label_map = {"Fully Paid": 0, "Charged Off": 1}

y_train = df_train_raw[TARGET].map(label_map)
y_test = df_test_raw[TARGET].map(label_map)

print(f"Distribucion train:{y_train.value_counts()}")
print(f"Distribucion test:{y_test.value_counts()}")


Distribucion train:loan_status
0    63764
1    16236
Name: count, dtype: int64
Distribucion test:loan_status
0    16003
1     3997
Name: count, dtype: int64


## 3. Preprocesamiento


Se aplica un preprocesamiento siguiendo el patron `fit/transform` para aprender las transformaciones solo en train y reutilizarlas exactamente igual en test.

Justificacion:
- Se eliminan variables con demasiados nulos.
- Se imputa por moda o mediana segun el porcentaje de valores perdidos.
- Se transforman variables categoricas a formato numerico apto para modelado.
- Se crean variables derivadas de FICO, deuda e intervalos de renta/patrimonio.
- Se usa `RobustScaler` para amortiguar el efecto de outliers.


In [5]:
raw_predictors_vars = (
    pd.read_excel("data/variables_withExperts.xlsx")
    .query("posible_predictora == 'si'")
    .variable
    .tolist()
)
print(f"Variables predictoras del Excel: {len(raw_predictors_vars)}")

df_train_sel = df_train_raw[raw_predictors_vars].copy()
df_test_sel = df_test_raw[raw_predictors_vars].copy()


Variables predictoras del Excel: 101


In [6]:
preproc = Practica1Preprocessing()
X_train_prep = preproc.fit_transform(df_train_sel, y_train)
X_test_prep = preproc.transform(df_test_sel)

assert X_train_prep.isnull().sum().sum() == 0
assert X_test_prep.isnull().sum().sum() == 0

print(f"Preprocessing: Train: {X_train_prep.shape} | Test: {X_test_prep.shape}")


Preprocessing: Train: (80000, 105) | Test: (20000, 105)


## 4. Filtering


Se aplica un filtrado posterior para reducir redundancia y ruido.

Justificacion:
- `DropCorrelatedFeatures` elimina variables muy correlacionadas.
- `VarianceThreshold` descarta variables casi constantes.
- `ProbeFeatureSelection` conserva variables superiores al ruido.


In [7]:
filtering = Practica1Filtering(
    corr_threshold=0.9,
    variance_threshold=0.95,
    n_probes=10,
    cv=3,
    random_state=42,
)

X_train = filtering.fit_transform(X_train_prep, y_train)
X_test = filtering.transform(X_test_prep)

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

print(f"Filtering: Train: {X_train.shape} | Test: {X_test.shape}")


Filtering: Train: (80000, 19) | Test: (20000, 19)


### Resumen de dimensiones


In [8]:
resumen = pd.DataFrame({
    "Etapa": [
        "Raw (cols predictoras)",
        "Tras Practica1Preprocessing",
        "Tras Practica1Filtering",
    ],
    "Train (filas x cols)": [str(df_train_sel.shape), str(X_train_prep.shape), str(X_train.shape)],
    "Test (filas x cols)": [str(df_test_sel.shape), str(X_test_prep.shape), str(X_test.shape)],
})
resumen


,Etapa,Train (filas x cols),Test (filas x cols)
0,Raw (cols predictoras),"(80000, 101)","(20000, 101)"
1,Tras Practica1Preprocessing,"(80000, 105)","(20000, 105)"
2,Tras Practica1Filtering,"(80000, 19)","(20000, 19)"


## 5. Entrenamiento de modelos


### 5.1 Funcion de evaluacion


In [9]:
resultados_lista = []

def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)

    if hasattr(modelo, "predict_proba"):
        y_score = modelo.predict_proba(X_te)[:, 1]
    else:
        y_score = modelo.decision_function(X_te)

    m = {
        "Modelo": nombre,
        "Accuracy": round(accuracy_score(y_te, y_pred), 4),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_te, y_pred, zero_division=0), 4),
        "PR-AUC": round(average_precision_score(y_te, y_score), 4),
    }

    print(f"{nombre}")
    print(classification_report(y_te, y_pred, target_names=["Fully Paid", "Charged Off"]))
    resultados_lista.append(m)
    return modelo


### 5.2 Gradient Boosting


In [10]:
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42,
)
evaluar_modelo("Gradient Boosting", gb_model, X_train, y_train, X_test, y_test)


Gradient Boosting
              precision    recall  f1-score   support

  Fully Paid       0.81      0.99      0.89     16003
 Charged Off       0.53      0.07      0.12      3997

    accuracy                           0.80     20000
   macro avg       0.67      0.53      0.50     20000
weighted avg       0.75      0.80      0.73     20000



,loss,'log_loss'
,learning_rate,0.05
,n_estimators,200
,subsample,0.8
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,4
,min_impurity_decrease,0.0
,init,None


### 5.3 SVM Model

In [11]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    random_state=42,
)
evaluar_modelo("SVM (RBF)", svm_model, X_train, y_train, X_test, y_test)


SVM (RBF)
              precision    recall  f1-score   support

  Fully Paid       0.87      0.63      0.74     16003
 Charged Off       0.30      0.63      0.41      3997

    accuracy                           0.63     20000
   macro avg       0.59      0.63      0.57     20000
weighted avg       0.76      0.63      0.67     20000



,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


### 5.4 MLP


In [12]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    batch_size=256,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
)
evaluar_modelo("MLP (Red neuronal)", mlp_model, X_train, y_train, X_test, y_test)


MLP (Red neuronal)
              precision    recall  f1-score   support

  Fully Paid       0.81      0.99      0.89     16003
 Charged Off       0.51      0.06      0.10      3997

    accuracy                           0.80     20000
   macro avg       0.66      0.52      0.49     20000
weighted avg       0.75      0.80      0.73     20000



,hidden_layer_sizes,"(128, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.001
,batch_size,256
,learning_rate,'adaptive'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,42


## 6. Comparacion con el modelo base


In [13]:
def evaluar_modelo_base_fico(df_te, y_te, threshold=0.67):
    fico_medio = (df_te["fico_range_low"] + df_te["fico_range_high"]) / 2
    fico_normalizado = (fico_medio - 300) / (850 - 300)
    y_pred = np.where(fico_normalizado > threshold, 0, 1)
    y_score = 1 - fico_normalizado

    return {
        "Modelo": "Modelo base FICO",
        "Accuracy": round(accuracy_score(y_te, y_pred), 4),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_te, y_pred, zero_division=0), 4),
        "PR-AUC": round(average_precision_score(y_te, y_score), 4),
    }

resultado_base = evaluar_modelo_base_fico(df_test_sel, y_test)
resultado_base


{'Modelo': 'Modelo base FICO',
 'Accuracy': 0.7166,
 'Precision': 0.2647,
 'Recall': 0.2352,
 'PR-AUC': 0.2442}

In [14]:
resultados = pd.concat(
    [pd.DataFrame([resultado_base]), pd.DataFrame(resultados_lista)],
    ignore_index=True,
).set_index("Modelo")
resultados


,Accuracy,Precision,Recall,PR-AUC
Modelo,,,,
Modelo base FICO,0.7166,0.2647,0.2352,0.2442
Gradient Boosting,0.8018,0.5335,0.0678,0.3650
SVM (RBF),0.6345,0.3023,0.6335,0.3366
MLP (Red neuronal),0.8007,0.5140,0.0550,0.3575


In [15]:
metricas = ["Accuracy", "Precision", "Recall", "PR-AUC"]
base_name = "Modelo base FICO"

comparativa_delta = resultados.copy()
for metrica in metricas:
    comparativa_delta[f"Delta {metrica}"] = comparativa_delta[metrica] - comparativa_delta.loc[base_name, metrica]

comparativa_delta


,Accuracy,Precision,Recall,PR-AUC,Delta Accuracy,Delta Precision,Delta Recall,Delta PR-AUC
Modelo,,,,,,,,
Modelo base FICO,0.7166,0.2647,0.2352,0.2442,0.0000,0.0000,0.0000,0.0000
Gradient Boosting,0.8018,0.5335,0.0678,0.3650,0.0852,0.2688,-0.1674,0.1208
SVM (RBF),0.6345,0.3023,0.6335,0.3366,-0.0821,0.0376,0.3983,0.0924
MLP (Red neuronal),0.8007,0.5140,0.0550,0.3575,0.0841,0.2493,-0.1802,0.1133


En comparación con el modelo base FICO, los modelos evaluados no presentan una mejora homogénea en todas las métricas. El modelo Gradient Boosting mejora la accuracy en 0,0852, la precision en 0,2688 y la PR-AUC en 0,1208, pero empeora el recall en 0,1674. De forma similar, el modelo MLP incrementa la accuracy en 0,0841, la precision en 0,2493 y la PR-AUC en 0,1133, aunque reduce el recall en 0,1802. Por tanto, ambos modelos superan claramente al modelo base en capacidad global de clasificación y en calidad de las predicciones positivas, pero muestran una menor capacidad para detectar la clase positiva.

Por otro lado, el modelo SVM (RBF) presenta un comportamiento diferente: empeora la accuracy en 0,0821, pero mejora la precision en 0,0376, el recall en 0,3983 y la PR-AUC en 0,0924. De hecho, es el modelo con mayor recall, lo que indica una mejor detección de positivos, aunque a costa de una menor exactitud global.

En consecuencia, no puede afirmarse que exista un modelo claramente superior al modelo base en todas las métricas. La elección del mejor modelo dependerá de la prioridad del problema: si se busca maximizar la detección de positivos, el SVM resulta más adecuado; si se priorizan la accuracy, la precision y la PR-AUC, destacan Gradient Boosting y MLP. En los casos en que un modelo no supera al base en una métrica concreta, esta situación puede justificarse por el umbral de decisión utilizado, el desbalance de clases, la selección de variables o una configuración no óptima de hiperparámetros. Como mejora, sería conveniente probar ajustes de umbral, técnicas de balanceo, una revisión de la selección de variables y una optimización más fina de hiperparámetros.